# Ek Animal Expert System implement karna

[AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) ka ek example.

Is sample mein hum kuch physical characteristics ke basis par kisi animal ko pehchaanne ke liye ek simple knowledge-based system implement karenge. System ko neeche diye gaye AND-OR tree se dikhaya ja sakta hai (yeh poore tree ka sirf ek hissa hai, hum aasaani se kuch aur rules add kar sakte hain):

![](../../../../../../translated_images/hi/AND-OR-Tree.5592d2c70187f283.webp)

## Backward Inference ke saath apna khud ka expert system shell

Chalo production rules par based knowledge representation ke liye ek simple language define karte hain. Rules define karne ke liye hum Python classes ko keywords ki tarah use karenge. Basically teen types ki classes hongi:
* `Ask` ek question ko represent karti hai jo user se poocha jaana hai. Isme possible answers ka set hota hai.
* `If` ek rule ko represent karti hai, aur yeh sirf rule ka content store karne ke liye ek syntactic sugar hai.
* `AND`/`OR` classes tree ki AND/OR branches ko represent karne ke liye hain. Yeh bas andar ke arguments ki list store karti hain. Code ko simple rakhne ke liye, saari functionality parent class `Content` mein define ki gayi hai.

In [ ]:
class Ask():
    def __init__(self,choices=['y','n']):
        self.choices = choices
    def ask(self):
        if max([len(x) for x in self.choices])>1:
            for i,x in enumerate(self.choices):
                print("{0}. {1}".format(i,x),flush=True)
            x = int(input())
            return self.choices[x]
        else:
            print("/".join(self.choices),flush=True)
            return input()

class Content():
    def __init__(self,x):
        self.x=x
        
class If(Content):
    pass

class AND(Content):
    pass

class OR(Content):
    pass

Hamare system mein, working memory mein **facts** ki list hogi, jo **attribute-value pairs** ke form mein hogi. Knowledge base ko ek bade dictionary ke roop mein define kiya ja sakta hai jo actions (naye facts jinhein working memory mein add karna hai) ko conditions se map karta hai, jo AND-OR expressions ke roop mein likhi gayi hain. Saath hi, kuch facts ko `ask` bhi kiya ja sakta hai.

In [ ]:
rules = {
    'default': Ask(['y','n']),
    'color' : Ask(['red-brown','black and white','other']),
    'pattern' : Ask(['dark stripes','dark spots']),
    'mammal': If(OR(['hair','gives milk'])),
    'carnivor': If(OR([AND(['sharp teeth','claws','forward-looking eyes']),'eats meat'])),
    'ungulate': If(['mammal',OR(['has hooves','chews cud'])]),
    'bird': If(OR(['feathers',AND(['flies','lies eggs'])])),
    'animal:monkey' : If(['mammal','carnivor','color:red-brown','pattern:dark spots']),
    'animal:tiger' : If(['mammal','carnivor','color:red-brown','pattern:dark stripes']),
    'animal:giraffe' : If(['ungulate','long neck','long legs','pattern:dark spots']),
    'animal:zebra' : If(['ungulate','pattern:dark stripes']),
    'animal:ostrich' : If(['bird','long nech','color:black and white','cannot fly']),
    'animal:pinguin' : If(['bird','swims','color:black and white','cannot fly']),
    'animal:albatross' : If(['bird','flies well'])
}

Backward inference karne ke liye, hum `Knowledgebase` class define karenge. Isme yeh shamil honge:
* Working `memory` - ek dictionary jo attributes ko values se map karti hai
* Knowledge base ke `rules`, upar bataye gaye format mein

Do main methods hain:
* `get` kisi attribute ki value lene ke liye hai, aur zarurat padne par inference karta hai. Jaise, `get('color')` kisi color slot ki value lega (zarurat ho toh poochega, aur baad mein use ke liye value ko working memory mein store kar dega). Agar hum `get('color:blue')` poochte hain, toh yeh color ke liye poochega, aur phir color ke basis par `y`/`n` value return karega.
* `eval` asli inference karta hai, yaani AND/OR tree ko traverse karta hai, sub-goals ko evaluate karta hai, wagairah.

In [ ]:
class KnowledgeBase():
    def __init__(self,rules):
        self.rules = rules
        self.memory = {}
        
    def get(self,name):
        if ':' in name:
            k,v = name.split(':')
            vv = self.get(k)
            return 'y' if v==vv else 'n'
        if name in self.memory.keys():
            return self.memory[name]
        for fld in self.rules.keys():
            if fld==name or fld.startswith(name+":"):
                # print(" + proving {}".format(fld))
                value = 'y' if fld==name else fld.split(':')[1]
                res = self.eval(self.rules[fld],field=name)
                if res!='y' and res!='n' and value=='y':
                    self.memory[name] = res
                    return res
                if res=='y':
                    self.memory[name] = value
                    return value
        # field is not found, using default
        res = self.eval(self.rules['default'],field=name)
        self.memory[name]=res
        return res
                
    def eval(self,expr,field=None):
        # print(" + eval {}".format(expr))
        if isinstance(expr,Ask):
            print(field)
            return expr.ask()
        elif isinstance(expr,If):
            return self.eval(expr.x)
        elif isinstance(expr,AND) or isinstance(expr,list):
            expr = expr.x if isinstance(expr,AND) else expr
            for x in expr:
                if self.eval(x)=='n':
                    return 'n'
            return 'y'
        elif isinstance(expr,OR):
            for x in expr.x:
                if self.eval(x)=='y':
                    return 'y'
            return 'n'
        elif isinstance(expr,str):
            return self.get(expr)
        else:
            print("Unknown expr: {}".format(expr))

Ab chalo apna animal knowledge base define karte hain aur consultation karte hain. Dhyan rakhiye ki yeh call aapse questions poochega. Yes-no questions ke liye aap `y`/`n` type karke answer de sakte hain, ya lambe multiple-choice answers wale questions ke liye number (0..N) de sakte hain.

In [ ]:
kb = KnowledgeBase(rules)
kb.get('animal')

## Forward Inference ke liye Experta ka use

Agle example mein hum knowledge representation ki libraries mein se ek, [Experta](https://github.com/nilp0inter/experta), use karke forward inference implement karne ki koshish karenge. **Experta** ek library hai jo Python mein forward inference systems banane ke liye banayi gayi hai, aur jise purane classic system [CLIPS](http://www.clipsrules.net/index.html) jaisa design kiya gaya hai.

Forward chaining hum khud bhi bina kisi badi dikkat ke implement kar sakte the, lekin general implementations aam taur par bahut efficient nahi hoti. Zyada effective rule matching ke liye ek special algorithm [Rete](https://en.wikipedia.org/wiki/Rete_algorithm) use hota hai.

In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/nilp0inter/experta

In [ ]:
from experta import *
#import experta

Hum apne system ko `KnowledgeEngine` ki subclass ke roop mein define karenge. Har rule ek alag function ke roop mein define hota hai jisme `@Rule` annotation hota hai, jo batata hai ki rule kab activate hona chahiye. Rule ke andar hum `declare` function use karke naye facts add kar sakte hain, aur un facts ko add karne se inference engine aage kuch aur rules ko activate kar dega.

In [ ]:
class Animals(KnowledgeEngine):
    @Rule(OR(
           AND(Fact('sharp teeth'),Fact('claws'),Fact('forward looking eyes')),
           Fact('eats meat')))
    def cornivor(self):
        self.declare(Fact('carnivor'))
        
    @Rule(OR(Fact('hair'),Fact('gives milk')))
    def mammal(self):
        self.declare(Fact('mammal'))

    @Rule(Fact('mammal'),
          OR(Fact('has hooves'),Fact('chews cud')))
    def hooves(self):
        self.declare('ungulate')
        
    @Rule(OR(Fact('feathers'),AND(Fact('flies'),Fact('lays eggs'))))
    def bird(self):
        self.declare('bird')
        
    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark spots'))
    def monkey(self):
        self.declare(Fact(animal='monkey'))

    @Rule(Fact('mammal'),Fact('carnivor'),
          Fact(color='red-brown'),
          Fact(pattern='dark stripes'))
    def tiger(self):
        self.declare(Fact(animal='tiger'))

    @Rule(Fact('ungulate'),
          Fact('long neck'),
          Fact('long legs'),
          Fact(pattern='dark spots'))
    def giraffe(self):
        self.declare(Fact(animal='giraffe'))

    @Rule(Fact('ungulate'),
          Fact(pattern='dark stripes'))
    def zebra(self):
        self.declare(Fact(animal='zebra'))

    @Rule(Fact('bird'),
          Fact('long neck'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def straus(self):
        self.declare(Fact(animal='ostrich'))

    @Rule(Fact('bird'),
          Fact('swims'),
          Fact('cannot fly'),
          Fact(color='black and white'))
    def pinguin(self):
        self.declare(Fact(animal='pinguin'))

    @Rule(Fact('bird'),
          Fact('flies well'))
    def albatros(self):
        self.declare(Fact(animal='albatross'))
        
    @Rule(Fact(animal=MATCH.a))
    def print_result(self,a):
          print('Animal is {}'.format(a))
                    
    def factz(self,l):
        for x in l:
            self.declare(x)

Ek baar knowledge base define ho jaaye, toh hum apni working memory ko kuch initial facts se bharte hain, aur phir inference karne ke liye `run()` method call karte hain. Result mein aap dekh sakte hain ki naye infer kiye gaye facts working memory mein add ho jaate hain, jisme animal ke baare mein final fact bhi shamil hai (agar humne saare initial facts sahi se set kiye hain).

In [ ]:
ex1 = Animals()
ex1.reset()
ex1.factz([
    Fact(color='red-brown'),
    Fact(pattern='dark stripes'),
    Fact('sharp teeth'),
    Fact('claws'),
    Fact('forward looking eyes'),
    Fact('gives milk')])
ex1.run()
ex1.facts

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->